# Module 1: Setup and Imports

In [ ]:
import os
import cv2
import shap
import timm
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import AutoConfig, AutoModel
from albumentations import (HorizontalFlip, RandomRotate90, ShiftScaleRotate,
                            ColorJitter, Resize, Compose)
from albumentations.pytorch import ToTensorV2
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Module 2: Data Preprocessing and Dataset Loader

In [ ]:

IMAGE_DIR = "/path/to/HAM10000/images"
CSV_PATH = "/path/to/HAM10000_metadata.csv"

def remove_hair(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, thresh = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    return cv2.inpaint(image, thresh, 1, cv2.INPAINT_TELEA)

def shades_of_gray(image, p=6):
    image = image.astype(np.float32)
    image = np.power(image, p)
    rgb_mean = np.power(np.mean(image, axis=(0, 1)), 1/p)
    norm = np.sqrt(np.sum(np.power(rgb_mean, 2.0)))
    rgb_mean = rgb_mean / norm
    image = image / rgb_mean
    return np.clip(image, 0, 255).astype(np.uint8)

def encode_metadata(df):
    df = df.copy()
    df['age'] = df['age'].fillna(df['age'].mean())
    df['sex'] = LabelEncoder().fit_transform(df['sex'].fillna('unknown'))
    df['site'] = LabelEncoder().fit_transform(df['site'].fillna('unknown'))
    scaler = MinMaxScaler()
    df['age'] = scaler.fit_transform(df[['age']])
    return df

train_transform = Compose([
    Resize(224, 224), HorizontalFlip(p=0.5),
    ShiftScaleRotate(0.1, 0.1, 30, p=0.5), ColorJitter(p=0.3), ToTensorV2()
])
val_transform = Compose([Resize(224, 224), ToTensorV2()])

class SkinLesionDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image_path'])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = remove_hair(image)
        image = shades_of_gray(image)
        metadata = np.array([row['age'], row['sex'], row['site']], dtype=np.float32)
        label = row['label']
        if self.transform:
            image = self.transform(image=image)['image']
        return image, torch.tensor(metadata), torch.tensor(label)
    

# Module 3: Dataset Loading and Splitting

In [ ]:

df = pd.read_csv(CSV_PATH)
df = df[['image_id', 'dx', 'age', 'sex', 'localization']]
df = df.rename(columns={'image_id': 'image_path', 'dx': 'label', 'localization': 'site'})
df['image_path'] = df['image_path'].apply(lambda x: x + ".jpg")
df = encode_metadata(df)
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

train_dataset = SkinLesionDataset(train_df, IMAGE_DIR, transform=train_transform)
val_dataset = SkinLesionDataset(val_df, IMAGE_DIR, transform=val_transform)
test_dataset = SkinLesionDataset(test_df, IMAGE_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

NUM_CLASSES = df['label'].nunique()
    

# Module 5: Focal Loss and Optimizer Setup

In [ ]:

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            loss *= alpha_t
        return loss.mean() if self.reduction == 'mean' else loss.sum()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SkinNetHDA(num_classes=NUM_CLASSES).to(device)

class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum()
alpha = class_weights.to(device)

criterion = FocalLoss(alpha=alpha, gamma=2.0)
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    

# Module 6: Training and Validation Loop

In [ ]:

from sklearn.metrics import roc_auc_score, accuracy_score

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, true_labels, pred_logits = 0, [], []
    for images, metadata, labels in tqdm(loader):
        images, metadata, labels = images.to(device), metadata.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images, metadata)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        true_labels.extend(labels.cpu().numpy())
        pred_logits.extend(outputs.detach().cpu().numpy())
    preds = np.argmax(pred_logits, axis=1)
    return total_loss / len(loader.dataset), accuracy_score(true_labels, preds), roc_auc_score(true_labels, pred_logits, multi_class='ovr')

def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss, true_labels, pred_logits = 0, [], []
    with torch.no_grad():
        for images, metadata, labels in loader:
            images, metadata, labels = images.to(device), metadata.to(device), labels.to(device)
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            true_labels.extend(labels.cpu().numpy())
            pred_logits.extend(outputs.cpu().numpy())
    preds = np.argmax(pred_logits, axis=1)
    return total_loss / len(loader.dataset), accuracy_score(true_labels, preds), roc_auc_score(true_labels, pred_logits, multi_class='ovr')

EPOCHS, best_auc, patience_counter = 30, 0, 0
for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")
    train_loss, train_acc, train_auc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_auc = validate_epoch(model, val_loader, criterion)
    scheduler.step()
    print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, AUC: {train_auc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, AUC: {val_auc:.4f}")
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_skinnet_hda.pth")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print("Early stopping.")
            break
    

# Module 7: Explainability with Grad-CAM++ and SHAP

In [ ]:

def show_gradcam(model, image_tensor, metadata_tensor, target_class, layer=None):
    model.eval()
    if layer is None:
        layer = model.backbone[-1]
    cam = GradCAMPlusPlus(model=model, target_layers=[layer], use_cuda=torch.cuda.is_available())
    input_tensor = image_tensor.unsqueeze(0).to(device)
    metadata_tensor = metadata_tensor.unsqueeze(0).to(device)
    def forward_func(x): return model(x, metadata_tensor)
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(target_class)], eigen_smooth=True)[0]
    rgb_image = image_tensor.permute(1, 2, 0).cpu().numpy()
    rgb_image = (rgb_image - rgb_image.min()) / (rgb_image.max() - rgb_image.min())
    cam_image = show_cam_on_image(rgb_image, grayscale_cam, use_rgb=True)
    plt.imshow(cam_image)
    plt.title(f"Grad-CAM++: Class {target_class}")
    plt.axis('off')
    plt.show()

def explain_shap(model, image_tensor, metadata_tensor):
    model.eval()
    def model_wrapper(meta_batch):
        image_batch = image_tensor.unsqueeze(0).repeat(len(meta_batch), 1, 1, 1).to(device)
        meta_batch = torch.tensor(meta_batch, dtype=torch.float32).to(device)
        return model(image_batch, meta_batch).detach().cpu().numpy()
    explainer = shap.Explainer(model_wrapper, masker=shap.maskers.Independent(metadata_tensor.numpy()))
    shap_values = explainer(metadata_tensor.numpy())
    shap.plots.bar(shap_values[0], feature_names=["age", "sex", "site"])
    

# Module 8: Final Evaluation on Test Set

In [ ]:

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns

def evaluate_on_test(model, loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for images, metadata, labels in tqdm(loader):
            images, metadata, labels = images.to(device), metadata.to(device), labels.to(device)
            outputs = model(images, metadata)
            probs = F.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(torch.argmax(probs, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

model.load_state_dict(torch.load("best_skinnet_hda.pth"))
y_true, y_pred, y_prob = evaluate_on_test(model, test_loader)

print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))
try:
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr')
    print(f"AUC Score (OVR): {auc:.4f}")
except:
    print("AUC computation failed.")

conf_mat = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap="Blues",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()
    